# Fase 3.1: Modelado - Aprendizaje supervisado

In [1]:
# Importar dependencias.
import constants
import gc
import lightgbm as lgb
import numpy as np
import optuna
import os
import pandas as pd
import tensorflow as tf
import warnings
from catboost import CatBoostRegressor, Pool
from joblib import dump
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.models import Sequential
from util import csv_util, date_util

# No mostrar mensajes de advertencia.
optuna.logging.set_verbosity(optuna.logging.ERROR)
warnings.filterwarnings('ignore')

I0000 00:00:1793189667.627262     295 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793189667.887692     295 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793189669.200179     295 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. Introducción

Un modelo de aprendizaje supervisado procesa un conjunto de datos etiquetados y, por tanto, su objetivo es clasificar datos nuevos o predecir resultados. Estos modelos se entrenará con el conjunto de datos de ambas ciudades. Además, es conveniente añadir nuevas características derivadas:
- Las medias móviles añaden tendencias al modelo. Por ejemplo: ¿cuál ha sido la tendencia de las últimas semanas con respecto a las temperaturas máximas o las precipitaciones?
- Los retrasos añaden memoria al modelo. Por ejemplo: ¿cuál ha sido la temperatura máxima o la precipitación de ayer?

In [2]:
def load_dataset(cyclic_coding=True):
    df = csv_util.read('dataset/processed_data/dataset.csv', usecols=['date', 'city', 'section', 'roadway', 'direction', 'lane', 'heavy_traffic', 'max_pavement_temperature', 'min_pavement_temperature',
                                                                      'uv_index', 'precipitation', 'total_stress'],
                       dtype={'city':str, 'section':int, 'roadway':int, 'direction':int, 'lane':int, 'heavy_traffic':float, 'max_pavement_temperature':float, 'min_pavement_temperature':float,
                              'uv_index':float, 'precipitation':float, 'total_stress':float},
                       date_formats=[('date', '%Y-%m-%d')])

    if cyclic_coding:
        # Añadir codificación cíclica para días de la semana y año.
        df = date_util.add_cyclic_coding(df)

    return df

def split_dataset(df):
    # Dividir conjunto de datos en entrenamiento, validación y pruebas.
    df_train = df[df['date'].dt.year < constants.VAL_BOUNDARY_YEAR]
    df_val = df[(df['date'].dt.year >= constants.VAL_BOUNDARY_YEAR) & (df['date'].dt.year < constants.TEST_BOUNDARY_YEAR)]
    df_test = df[df['date'].dt.year >= constants.TEST_BOUNDARY_YEAR]

    return df_train, df_val, df_test

def split_X_y(df_train, df_val, df_test):
    target = 'total_stress'
    cols_to_exclude = ['date', 'section', 'roadway', 'direction', 'lane', target]
    features = [col for col in df_train.columns if col not in cols_to_exclude]

    X_train, y_train = df_train[features], df_train[target]
    X_val, y_val = df_val[features], df_val[target]
    X_test, y_test = df_test[features], df_test[target]

    return X_train, y_train, X_val, y_val, X_test, y_test

def add_rollings_and_lags(df):
    keys_by_lane = ['city', 'section', 'roadway', 'direction', 'lane']
    target_col = f'target_stress_{constants.HORIZON_DAYS}d'

    # Ordenar por carril y fecha para características individuales.
    df = df.sort_values(by=keys_by_lane + ['date']).reset_index(drop=True)

    # Identificar variable categórica.
    df['city'] = df['city'].astype('category')

    # Cálculos por carril.
    for day in [7, 30, 60]:
        df[f'heavy_traffic_rolling_{day}d'] = df.groupby(keys_by_lane)['heavy_traffic'].transform(lambda x: x.rolling(day).mean())
        df[f'heavy_traffic_lag_{day}'] = df.groupby(keys_by_lane)['heavy_traffic'].shift(day)

    df[f'heavy_traffic_lag_365'] = df.groupby(keys_by_lane)['heavy_traffic'].shift(365)
    # Calcular daño futuro.
    df[target_col] = df.groupby(keys_by_lane)['total_stress'].transform(lambda x: x.cumsum().shift(-(constants.HORIZON_DAYS - 1)) - x.cumsum().shift(1).fillna(0))

    # Crear subconjunto por ciudad y fecha para características globales.
    climate_cols = ['city', 'date', 'max_pavement_temperature', 'precipitation']
    df_climate = df[climate_cols].drop_duplicates(subset=['city', 'date']).sort_values(['city', 'date'])

    # Calcular medias e históricos por ciudad.
    for day in [7, 30, 60]:
        df_climate[f'max_pavement_temperature_rolling_{day}d'] = df_climate.groupby('city')['max_pavement_temperature'].transform(lambda x: x.rolling(day).mean())
        df_climate[f'max_pavement_temperature_lag_{day}'] = df_climate.groupby('city')['max_pavement_temperature'].shift(day)
        df_climate[f'precipitation_rolling_{day}d'] = df_climate.groupby('city')['precipitation'].transform(lambda x: x.rolling(day).sum())
        df_climate[f'precipitation_lag_{day}'] = df_climate.groupby('city')['precipitation'].shift(day)

    df_climate[f'max_pavement_temperature_lag_365'] = df_climate.groupby('city')['max_pavement_temperature'].shift(365)
    df_climate[f'precipitation_lag_365'] = df_climate.groupby('city')['precipitation'].shift(365)

    # Unir el subconjunto anterior al conjunto principal.
    climate_feature_cols = [col for col in df_climate.columns if col not in ['max_pavement_temperature', 'precipitation']]
    df = df.merge(df_climate[climate_feature_cols], on=['city', 'date'], how='left')

    # Eliminar filas con valores nulos generados por el desplazamiento del horizonte futuro (más allá del último año con datos).
    df = df.dropna(subset=[target_col])

    return df

def evaluate_model(model, X_test, y_test, is2D=False, scaler=None):
    y_pred = model.predict(X_test).flatten() if is2D else model.predict(X_test)

    # Solo si los datos están escalados.
    if scaler is not None:
        y_test = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
        y_pred = scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
        mre = np.mean(np.abs(y_test - y_pred) / np.maximum(np.abs(y_test), 1e-5))
        print(f'MRE: {mre:.4f}')
    else:
        wape = np.sum(np.abs(y_test - y_pred)) / np.maximum(np.sum(np.abs(y_test)), 1e-5)
        print(f'WAPE: {wape:.4f}')

    mae = mean_absolute_error(y_test, y_pred)
    mae_std = np.std(np.abs(y_test - y_pred))
    rmse = root_mean_squared_error(y_test, y_pred)

    print(f'MAE: {mae:.4f} ± {mae_std:.4f}')
    print(f'RMSE: {rmse:.4f}')

    return y_pred

def save_model(model, model_name):
    path = f'model/{model_name}/trial/{model_name}.pkl'

    if os.path.exists(path):
        os.remove(path)

    dump(model, path)

def save_results(df_test, y_pred, model_name):
    if len(df_test) == len(y_pred):
        # Cuando hay predicciones para todos los días.
        pred_col = 'total_stress_pred'
        df_test[pred_col] = y_pred
        df_test = df_test[['date', 'city', 'section', 'roadway', 'direction', 'lane', 'total_stress', pred_col]]
    else:
        target_col = f'target_stress_{constants.HORIZON_DAYS}d'
        df_test[target_col] = np.nan
        pred_col = f'target_stress_{constants.HORIZON_DAYS}d_pred'
        df_test[pred_col] = np.nan
        keys_by_lane = ['city', 'section', 'roadway', 'direction', 'lane']

        # Agrupar por cada combinación única de carril.
        groups = list(df_test.groupby(keys_by_lane))

        # Para recorrer la lista de predicciones.
        y_pred_idx = 0

        for _, group in groups:
            group_len = len(group)
            n_windows = group_len - constants.HORIZON_DAYS + 1 - constants.WINDOW_SIZE

            if n_windows > 0:
                # Predicciones reales de este carril sin pasarnos del tamaño del grupo ni quedarnos sin elementos en la lista.
                valid_count = min(n_windows, len(y_pred) - y_pred_idx)

                # Calcular el acumulado real hacia adelante.
                if valid_count > 0:
                    real_sums = []
                    target_values = group['total_stress'].values

                    for i in range(constants.WINDOW_SIZE, constants.WINDOW_SIZE + valid_count):
                        # Suma desde el día actual 'i' durante los siguientes HORIZON_DAYS.
                        window_sum = np.sum(target_values[i : i + constants.HORIZON_DAYS])
                        real_sums.append(window_sum)

                    # Asignar los valores reales calculados.
                    df_test.loc[group.index[constants.WINDOW_SIZE : constants.WINDOW_SIZE + valid_count], target_col] = real_sums

                    # Asignar las predicciones del modelo.
                    df_test.loc[group.index[constants.WINDOW_SIZE : constants.WINDOW_SIZE + valid_count], pred_col] = y_pred[y_pred_idx : y_pred_idx + valid_count]
                    y_pred_idx += valid_count

        df_test = df_test[['date', 'city', 'section', 'roadway', 'direction', 'lane', 'total_stress', target_col, pred_col]]

    csv_util.save(df_test, f'model/{model_name}/result/{model_name}_results.csv', mode='w')

## 2. LightGBM

In [3]:
def objective_light_gbm(trial, X_train, y_train, X_val, y_val):
    # Hiperparámetros a optimizar.
    n_estimators = trial.suggest_int('n_estimators', 200, constants.MAX_ITERATIONS, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    num_leaves = trial.suggest_int('num_leaves', 20, 64)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True)
    reg_lambda = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True)

    lgbm_model = lgb.LGBMRegressor(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, num_leaves=num_leaves,
                                   subsample=subsample, colsample_bytree=colsample_bytree, reg_alpha=reg_alpha, reg_lambda=reg_lambda,
                                   objective='mae', metric='l1', random_state=42,
                                   n_jobs=-1, verbosity=-1, device='CPU')
    lgbm_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                   callbacks=[lgb.early_stopping(stopping_rounds=constants.PATIENCE, verbose=False)])
    mae = mean_absolute_error(y_val, lgbm_model.predict(X_val))
    print(f'Trial: {trial.number + 1}/{constants.NUM_TRIALS} | MAE: {mae:.4f}')

    # Eliminar la instancia del modelo. Limpiar la caché de la GPU.
    del lgbm_model
    gc.collect()

    return mae

def fit_light_gbm(study, X_train, y_train, X_val, y_val):
    best_params = study.best_params
    lgbm_model = lgb.LGBMRegressor(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'],
                                   num_leaves=best_params['num_leaves'], subsample=best_params['subsample'], colsample_bytree=best_params['colsample_bytree'],
                                   reg_alpha=best_params['reg_alpha'], reg_lambda=best_params['reg_lambda'], objective='mae', metric='l1', random_state=42, n_jobs=-1, verbosity=-1, device='CPU')
    lgbm_model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                   callbacks=[lgb.early_stopping(stopping_rounds=constants.PATIENCE, verbose=False)])

    # Estudiar sobreajuste.
    wape_train = np.sum(np.abs(y_train - lgbm_model.predict(X_train))) / np.maximum(np.sum(np.abs(y_train)), 1e-5)
    wape_val = np.sum(np.abs(y_val - lgbm_model.predict(X_val))) / np.maximum(np.sum(np.abs(y_val)), 1e-5)
    print(f'Métricas: Train (WAPE): {wape_train:.4f} | Val (WAPE): {wape_val:.4f}')

    return lgbm_model

In [4]:
print('LightGBM')
print('1. Añadir características temporales')
df = add_rollings_and_lags(load_dataset())

print('2. Dividir conjunto de datos en entrenamiento, validación y pruebas')
df_train, df_val, df_test = split_dataset(df)
X_train, y_train, X_val, y_val, X_test, y_test = split_X_y(df_train, df_val, df_test)

print('3. Buscar mejores hiperparámetros con Optuna')
study = optuna.create_study(direction='minimize')
study.optimize(lambda trial: objective_light_gbm(trial, X_train, y_train, X_val, y_val), n_trials=constants.NUM_TRIALS)

print('4. Entrenar con los mejores hiperparámetros')
lgbm_model = fit_light_gbm(study, X_train, y_train, X_val, y_val)
save_model(lgbm_model, 'LightGBM')

print('5. Evaluar modelo')
y_pred = evaluate_model(lgbm_model, X_test, y_test)

print('6. Guardar predicciones')
save_results(df_test, y_pred, 'LightGBM')

# Eliminar la instancia del modelo. Limpiar la caché de la GPU.
del lgbm_model
gc.collect()
print('FIN LightGBM')

LightGBM
1. Añadir características temporales
2. Dividir conjunto de datos en entrenamiento, validación y pruebas
3. Buscar mejores hiperparámetros con Optuna
Trial: 1/20 | MAE: 2938.5300
Trial: 2/20 | MAE: 3578.5928
Trial: 3/20 | MAE: 4092.2560
Trial: 4/20 | MAE: 2245.9879
Trial: 5/20 | MAE: 1899.8397
Trial: 6/20 | MAE: 2712.4306
Trial: 7/20 | MAE: 1938.3494
Trial: 8/20 | MAE: 2527.9766
Trial: 9/20 | MAE: 2025.5957
Trial: 10/20 | MAE: 2068.6346
Trial: 11/20 | MAE: 1956.1656
Trial: 12/20 | MAE: 1860.2578
Trial: 13/20 | MAE: 2027.7241
Trial: 14/20 | MAE: 1720.1215
Trial: 15/20 | MAE: 1760.7770
Trial: 16/20 | MAE: 1839.1079
Trial: 17/20 | MAE: 1898.6061
Trial: 18/20 | MAE: 1801.4293
Trial: 19/20 | MAE: 1773.3163
Trial: 20/20 | MAE: 2062.2482
4. Entrenar con los mejores hiperparámetros
Métricas: Train (WAPE): 0.0406 | Val (WAPE): 0.0684
5. Evaluar modelo
WAPE: 0.0705
MAE: 1937.7610 ± 3434.8786
RMSE: 3943.7683
6. Guardar predicciones
FIN LightGBM


## 3. CatBoost

In [5]:
def create_pools(X_train, y_train, X_val, y_val, X_test, y_test, cat_features):
    pool_train = Pool(X_train, y_train, cat_features=cat_features)
    pool_val = Pool(X_val, y_val, cat_features=cat_features)
    pool_test = Pool(X_test, y_test, cat_features=cat_features)

    return pool_train, pool_val, pool_test

def objective_cat_boost(trial, pool_train, pool_val):
    # Hiperparámetros a optimizar.
    iterations = trial.suggest_int('iterations', 500, constants.MAX_ITERATIONS, step=100)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    depth = trial.suggest_int('depth', 6, 10, step=1)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True)

    cat_boost_model = CatBoostRegressor(iterations=iterations, learning_rate=learning_rate, depth=depth, l2_leaf_reg=l2_leaf_reg,
                                        loss_function='RMSE', eval_metric='RMSE', early_stopping_rounds=constants.PATIENCE, random_seed=42,
                                        verbose=0, task_type='GPU')
    cat_boost_model.fit(pool_train, eval_set=pool_val, use_best_model=True)
    best_score = cat_boost_model.get_best_score()
    rmse = best_score['validation']['RMSE']
    print(f'Trial: {trial.number + 1}/{constants.NUM_TRIALS} | RMSE: {rmse:.4f}')

    # Eliminar la instancia del modelo. Limpiar la caché de la GPU.
    del cat_boost_model
    gc.collect()

    return rmse

def fit_cat_boost(study, pool_train, y_train, pool_val, y_val):
    best_params = study.best_params
    cat_boost_model = CatBoostRegressor(iterations=best_params['iterations'], learning_rate=best_params['learning_rate'], depth=best_params['depth'],
                                        l2_leaf_reg=best_params['l2_leaf_reg'], loss_function='RMSE', eval_metric='RMSE', early_stopping_rounds=constants.PATIENCE,
                                        random_seed=42, verbose=0, task_type='GPU')
    cat_boost_model.fit(pool_train, eval_set=pool_val, use_best_model=True)

    # Estudiar sobreajuste.
    wape_train = np.sum(np.abs(y_train - cat_boost_model.predict(pool_train))) / np.maximum(np.sum(np.abs(y_train)), 1e-5)
    wape_val = np.sum(np.abs(y_val - cat_boost_model.predict(pool_val))) / np.maximum(np.sum(np.abs(y_val)), 1e-5)
    print(f'Métricas: Train (WAPE): {wape_train:.4f} | Val (WAPE): {wape_val:.4f}')

    return cat_boost_model

In [6]:
print('CatBoost')
print('1. Añadir características temporales')
df = add_rollings_and_lags(load_dataset())

print('2. Dividir conjunto de datos en entrenamiento, validación y pruebas')
df_train, df_val, df_test = split_dataset(df)
X_train, y_train, X_val, y_val, X_test, y_test = split_X_y(df_train, df_val, df_test)
pool_train, pool_val, pool_test = create_pools(X_train, y_train, X_val, y_val, X_test, y_test, ['city'])

print('3. Buscar mejores hiperparámetros con Optuna')
study = optuna.create_study(direction='minimize')
study.optimize(lambda trial: objective_cat_boost(trial, pool_train, pool_val), n_trials=constants.NUM_TRIALS)

print('4. Entrenar con los mejores hiperparámetros')
cat_boost_model = fit_cat_boost(study, pool_train, y_train, pool_val, y_val)
save_model(cat_boost_model, 'CatBoost')

print('5. Evaluar modelo')
y_pred = evaluate_model(cat_boost_model, pool_test, y_test)

print('6. Guardar predicciones')
save_results(df_test, y_pred, 'CatBoost')

# Eliminar la instancia del modelo. Limpiar la caché de la GPU.
del cat_boost_model
gc.collect()
print('FIN CatBoost')

CatBoost
1. Añadir características temporales
2. Dividir conjunto de datos en entrenamiento, validación y pruebas
3. Buscar mejores hiperparámetros con Optuna
Trial: 1/20 | RMSE: 4405.3253
Trial: 2/20 | RMSE: 4224.5482
Trial: 3/20 | RMSE: 3115.5945
Trial: 4/20 | RMSE: 3543.3816
Trial: 5/20 | RMSE: 3889.7338
Trial: 6/20 | RMSE: 4518.6857
Trial: 7/20 | RMSE: 3212.5016
Trial: 8/20 | RMSE: 3603.9037
Trial: 9/20 | RMSE: 3214.0657
Trial: 10/20 | RMSE: 3240.8586
Trial: 11/20 | RMSE: 3214.4405
Trial: 12/20 | RMSE: 3432.1840
Trial: 13/20 | RMSE: 3066.2016
Trial: 14/20 | RMSE: 3115.7139
Trial: 15/20 | RMSE: 3224.3216
Trial: 16/20 | RMSE: 3102.2673
Trial: 17/20 | RMSE: 3183.6559
Trial: 18/20 | RMSE: 3182.4545
Trial: 19/20 | RMSE: 3110.9261
Trial: 20/20 | RMSE: 3057.2389
4. Entrenar con los mejores hiperparámetros
Métricas: Train (WAPE): 0.0322 | Val (WAPE): 0.0704
5. Evaluar modelo
WAPE: 0.0680
MAE: 1868.8050 ± 2945.3358
RMSE: 3488.1851
6. Guardar predicciones
FIN CatBoost


## 4. LSTM

In [10]:
def apply_scale(df_train, df_val, df_test):
    # Estandarizar características con valores que siguen una distribución normal, donde el centro es la media.
    standard_scaler = StandardScaler()
    standard_scaler_cols = ['heavy_traffic', 'max_pavement_temperature', 'min_pavement_temperature']
    df_train[standard_scaler_cols] = standard_scaler.fit_transform(df_train[standard_scaler_cols])
    df_val[standard_scaler_cols] = standard_scaler.transform(df_val[standard_scaler_cols])
    df_test[standard_scaler_cols] = standard_scaler.transform(df_test[standard_scaler_cols])

    # Normalizar características con valores con límites bien definidos.
    min_max_scaler = MinMaxScaler()
    min_max_scaler_cols = ['uv_index']
    df_train[min_max_scaler_cols] = min_max_scaler.fit_transform(df_train[min_max_scaler_cols])
    df_val[min_max_scaler_cols] = min_max_scaler.transform(df_val[min_max_scaler_cols])
    df_test[min_max_scaler_cols] = min_max_scaler.transform(df_test[min_max_scaler_cols])

    # Escalar características con muchos valores iguales a 0.
    robust_scaler = RobustScaler()
    robust_scaler_cols = ['precipitation']
    df_train[robust_scaler_cols] = robust_scaler.fit_transform(df_train[robust_scaler_cols])
    df_val[robust_scaler_cols] = robust_scaler.transform(df_val[robust_scaler_cols])
    df_test[robust_scaler_cols] = robust_scaler.transform(df_test[robust_scaler_cols])   

    return df_train, df_val, df_test

def apply_scale_to_target(y_train, y_val, y_test):
    stress_scaler = StandardScaler()
    y_train_scaled = stress_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_val_scaled = stress_scaler.transform(y_val.reshape(-1, 1)).flatten()
    y_test_scaled = stress_scaler.transform(y_test.reshape(-1, 1)).flatten()

    return y_train_scaled, y_val_scaled, y_test_scaled, stress_scaler

def create_windows(df):
    features = ['heavy_traffic', 'precipitation', 'uv_index', 'min_pavement_temperature', 'max_pavement_temperature']
    target_col = 'total_stress'
    X_list, y_list = [], []

    # Agrupar estrictamente por carril para respetar la integridad espacial y temporal.
    for _, group in df.groupby(['city', 'section', 'roadway', 'direction', 'lane']):
        group = group.sort_values('date')
        feature_values = group[features].values
        target_values = group[target_col].values

        # Crear ventanas deslizantes ajustando el límite para no salir del horizonte acumulado.
        for i in range(constants.WINDOW_SIZE, len(group) - constants.HORIZON_DAYS + 1):
            X_list.append(feature_values[i - constants.WINDOW_SIZE : i])
            # Calcular daño acumulado según el horizonte temporal.
            y_list.append(np.sum(target_values[i : i + constants.HORIZON_DAYS]))

    X_windows = np.array(X_list).astype('float32')
    y_windows = np.array(y_list).astype('float32')

    return X_windows, y_windows

def objective_lstm(trial, X_train, y_train, X_val, y_val):
    # Hiperparámetros a optimizar.
    units_1 = trial.suggest_categorical('units_1', [32, 64, 128])
    units_2 = trial.suggest_categorical('units_2', [16, 32, 64])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4, step=0.1)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256])

    lstm_model = Sequential([LSTM(units_1, activation='tanh', return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
                             Dropout(dropout_rate),
                             LSTM(units_2, activation='tanh', return_sequences=False),
                             Dropout(dropout_rate),
                             Dense(16, activation='relu'),
                             Dense(1)])
    lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss='mae', metrics=['mae'])
    early_stopping = EarlyStopping(monitor='val_loss', patience=constants.PATIENCE_OPTUNA, restore_best_weights=True, verbose=0)
    history = lstm_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=constants.NUM_EPOCHS_OPTUNA, batch_size=batch_size, callbacks=[early_stopping], verbose=0)

    mae = min(history.history['val_loss'])
    print(f'Trial: {trial.number + 1}/{constants.NUM_TRIALS} | MAE: {mae:.4f}')

    # Eliminar la instancia del modelo. Limpiar la caché de la GPU.
    del lstm_model
    tf.keras.backend.clear_session()
    gc.collect()

    return mae

def fit_lstm(study, X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled, stress_scaler):
    best_params = study.best_params
    lstm_model = Sequential([LSTM(best_params['units_1'], activation='tanh', return_sequences=True, input_shape=(X_train_scaled.shape[1], X_train_scaled.shape[2])),
                             Dropout(best_params['dropout_rate']),
                             LSTM(best_params['units_2'], activation='tanh', return_sequences=False),
                             Dropout(best_params['dropout_rate']),
                             Dense(16, activation='relu'),
                             Dense(1)])
    lstm_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']), loss='mae', metrics=['mae'])
    early_stopping = EarlyStopping(monitor='val_loss', patience=constants.PATIENCE, restore_best_weights=True, verbose=0)
    history = lstm_model.fit(X_train_scaled, y_train_scaled, validation_data=(X_val_scaled, y_val_scaled),
                             epochs=constants.NUM_EPOCHS_LSTM, batch_size=best_params['batch_size'], callbacks=[early_stopping], verbose=2)

    # Desestandarizar predicciones.
    y_pred_train_scaled = lstm_model.predict(X_train_scaled, verbose=0).flatten()
    y_pred_train = stress_scaler.inverse_transform(y_pred_train_scaled.reshape(-1, 1)).flatten()
    y_train = stress_scaler.inverse_transform(y_train_scaled.reshape(-1, 1)).flatten()
    y_pred_val_scaled = lstm_model.predict(X_val_scaled, verbose=0).flatten()
    y_pred_val = stress_scaler.inverse_transform(y_pred_val_scaled.reshape(-1, 1)).flatten()   
    y_val = stress_scaler.inverse_transform(y_val_scaled.reshape(-1, 1)).flatten()

    mre_train = np.mean(np.abs(y_train - y_pred_train) / np.maximum(np.abs(y_train), 1e-5))
    mre_val = np.mean(np.abs(y_val - y_pred_val) / np.maximum(np.abs(y_val), 1e-5))
    print(f'Métricas: Train (MRE): {mre_train:.4f} | Val (MRE): {mre_val:.4f}')

    return lstm_model

In [11]:
print('LSTM')
print('1. Dividir conjunto de datos en entrenamiento, validación y pruebas')
df_train, df_val, df_test = split_dataset(load_dataset(False))

print('2. Normalizar/estandarizar datos numéricos')
df_test_copied = df_test.copy()
df_train, df_val, df_test = apply_scale(df_train, df_val, df_test)

print('3. Crear ventanas temporales')
X_train, y_train = create_windows(df_train)
X_val, y_val = create_windows(df_val)
X_test, y_test = create_windows(df_test)
y_train_scaled, y_val_scaled, y_test_scaled, stress_scaler = apply_scale_to_target(y_train, y_val, y_test)

print('4. Buscar mejores hiperparámetros con Optuna')
study = optuna.create_study(direction='minimize')
study.optimize(lambda trial: objective_lstm(trial, X_train, y_train_scaled, X_val, y_val_scaled), n_trials=constants.NUM_TRIALS)

print('5. Entrenar con los mejores hiperparámetros')
lstm_model = fit_lstm(study, X_train, y_train_scaled, X_val, y_val_scaled, stress_scaler)
save_model(lstm_model, 'LSTM')

print('6. Evaluar modelo')
y_pred = evaluate_model(lstm_model, X_test, y_test_scaled, is2D=True, scaler=stress_scaler)

print('7. Guardar predicciones')
save_results(df_test_copied, y_pred, 'LSTM')

# Eliminar la instancia del modelo. Limpiar la caché de la GPU.
del lstm_model
tf.keras.backend.clear_session()
gc.collect()
print('FIN LSTM')

LSTM
1. Dividir conjunto de datos en entrenamiento, validación y pruebas
2. Normalizar/estandarizar datos numéricos
3. Crear ventanas temporales
4. Buscar mejores hiperparámetros con Optuna


I0000 00:00:1793156450.913089     114 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9705 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6
I0000 00:00:1793156453.745726   13885 cuda_dnn.cc:461] Loaded cuDNN version 92600


Trial: 1/20 | MAE: 0.2781
Trial: 2/20 | MAE: 0.2560
Trial: 3/20 | MAE: 0.2213
Trial: 4/20 | MAE: 0.2722
Trial: 5/20 | MAE: 0.2186
Trial: 6/20 | MAE: 0.2287
Trial: 7/20 | MAE: 0.2131
Trial: 8/20 | MAE: 0.2244
Trial: 9/20 | MAE: 0.2479
Trial: 10/20 | MAE: 0.2302
Trial: 11/20 | MAE: 0.2345
Trial: 12/20 | MAE: 0.2009
Trial: 13/20 | MAE: 0.2194
Trial: 14/20 | MAE: 0.2270
Trial: 15/20 | MAE: 0.2371
Trial: 16/20 | MAE: 0.2357
Trial: 17/20 | MAE: 0.2312
Trial: 18/20 | MAE: 0.2582
Trial: 19/20 | MAE: 0.2151
Trial: 20/20 | MAE: 0.2054
5. Entrenar con los mejores hiperparámetros
Epoch 1/80
304/304 - 5s - 17ms/step - loss: 0.2841 - mae: 0.2841 - val_loss: 0.2385 - val_mae: 0.2385
Epoch 2/80
304/304 - 4s - 14ms/step - loss: 0.2315 - mae: 0.2315 - val_loss: 0.2085 - val_mae: 0.2085
Epoch 3/80
304/304 - 4s - 14ms/step - loss: 0.1954 - mae: 0.1954 - val_loss: 0.2298 - val_mae: 0.2298
Epoch 4/80
304/304 - 4s - 14ms/step - loss: 0.1631 - mae: 0.1631 - val_loss: 0.2374 - val_mae: 0.2374
Epoch 5/80
304/30